<a href="https://colab.research.google.com/github/FeLocci/senacai/blob/main/spectogram_de_analise_de_som_desafio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

\#**Inicio do código**

\#Download do dataset

In [ ]:
import kagglehub
import os
import numpy as np
import librosa
import librosa.display
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from sklearn.model_selection import train_test_split
import os
import pandas as pd # Import pandas for reading metadata
import random

# 1. DOWNLOAD
path = kagglehub.dataset_download("soumendraprasad/musical-instruments-sound-dataset")
# The actual audio files for training are in a subfolder structure, typically:
# <download_path>/Train_submission/Train_submission/
# The metadata (labels) are in:
# <download_path>/Train_submission/Metadata_Train.csv

# Print path information for debugging
print("Dataset downloaded to:", path)
print("Content of download directory:", os.listdir(path))
print("Content of Train_submission directory:", os.listdir(os.path.join(path, "Train_submission/Train_submission")))
print(os.listdir(os.path.join(path, "Train_submission")))


\# Funcão para transformar o som em um espectograma

\# Modificar a variável duration para a duração do áudio usado

In [ ]:
def audio_to_spectrogram(audio_path):
    # Carrega 10 segundos
    y, sr = librosa.load(audio_path, duration=10)

    if len(y) == 0 or np.all(y == 0):
        return tf.zeros(IMG_SIZE + (3,), dtype=tf.float32)

    # Aumentando n_mels para 256 para maior resolução de frequência
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=256)
    S_dB = librosa.power_to_db(S + 1e-8, ref=np.max, top_db=80.0)

    min_val = np.min(S_dB)
    max_val = np.max(S_dB)

    if max_val == min_val:
        normalized_S_dB = np.zeros_like(S_dB)
    else:
        normalized_S_dB = (S_dB - min_val) / (max_val - min_val)

    img = tf.image.resize(np.expand_dims(normalized_S_dB, -1), IMG_SIZE)
    img = tf.image.grayscale_to_rgb(img)
    return img

\# Função de aumento


In [ ]:
# --- Data Augmentation Function ---
def augment_spectrogram_image(spec_image, max_noise_std=0.05):
    # Convert to TensorFlow tensor if not already
    if not tf.is_tensor(spec_image):
        spec_image = tf.convert_to_tensor(spec_image, dtype=tf.float32)

    # Add Gaussian noise
    noise = tf.random.normal(shape=tf.shape(spec_image), mean=0.0, stddev=tf.random.uniform([], 0, max_noise_std), dtype=tf.float32)
    augmented_image = spec_image + noise

    # Clip values to stay within [0, 1] range (assuming original spectro is normalized)
    augmented_image = tf.clip_by_value(augmented_image, 0.0, 1.0)

    return augmented_image.numpy() # Convert back to numpy for consistency
# --- End Data Augmentation Function ---

\# Cria as bases de teste e treino\
\# Houve dificuldades entre distinguir os sons de batria e de violino\
\# Adicionado a chamada a uma função de aumento do dado

In [ ]:
# Configurações de Memória e Dados
IMG_SIZE = (224, 224)
CLASSES = ['Sound_Guitar', 'Sound_Drum', 'Sound_Violin', 'Sound_Piano']
MAX_FILES_PER_CLASS = 150 # Aumentado para dar mais exemplos ao modelo

audio_files_base_path = os.path.join(path, "Train_submission", "Train_submission")
metadata_file_path = os.path.join(path, "Metadata_Train.csv")

x_data, y_data = [], []
label_to_int = {label: i for i, label in enumerate(CLASSES)}

if os.path.exists(metadata_file_path):
    metadata_df = pd.read_csv(metadata_file_path)

    for label_str in CLASSES:
        class_df = metadata_df[metadata_df['Class'] == label_str].head(MAX_FILES_PER_CLASS)
        print(f"Processando {len(class_df)} arquivos para {label_str}...")

        for index, row in class_df.iterrows():
            full_audio_path = os.path.join(audio_files_base_path, row['FileName'])
            if not os.path.exists(full_audio_path): continue

            try:
                spec = audio_to_spectrogram(full_audio_path)
                x_data.append(spec.numpy())
                y_data.append(label_to_int[label_str])

                # Aumentação sistemática para todas as classes
                # 3 variações por áudio original
                for _ in range(3):
                    augmented_spec = augment_spectrogram_image(spec, max_noise_std=0.08)
                    x_data.append(augmented_spec)
                    y_data.append(label_to_int[label_str])
            except:
                continue

X = np.array(x_data)
y = np.array(y_data)

if len(X) > 0:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(f"\nTotal de imagens (com aumentação): {len(X)}")
    print(f"Treino: {X_train.shape}, Teste: {X_test.shape}")

\# Treinamento do modelo

In [ ]:
class LossRatioCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        loss = logs.get('loss')
        val_loss = logs.get('val_loss')
        if loss is not None and val_loss is not None:
            # Calculate ratio: val_loss / loss
            ratio = val_loss / loss
            print(f' - val/train loss ratio: {ratio:.4f}')

class AccuracyRatioCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        accuracy = logs.get('accuracy')
        val_accuracy = logs.get('val_accuracy')
        if accuracy is not None and val_accuracy is not None:
            # Calculate ratio: val_loss / loss
            ratio = val_accuracy / accuracy
            print(f' - val/train accuracy ratio: {ratio:.4f}')


In [ ]:
import tensorflow as tf
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  print('GPU não encontrada. Vá em Ambiente de Execução > Alterar tipo para ativar a T4 GPU.')
else:
  print(f'Sucesso! GPU encontrada em: {device_name}')

In [ ]:
tf.keras.backend.clear_session()

base_model = applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.SpatialDropout2D(0.2), # Ajuda a evitar overfitting em características específicas do espectrograma
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(CLASSES), activation='softmax')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    verbose=1,
    restore_best_weights=True
)

print("Iniciando treinamento com maior resolução espectral...")
history = model.fit(
    X_train, y_train,
    epochs=40,
    validation_data=(X_test, y_test),
    batch_size=64,
    verbose=1,
    callbacks=[early_stop, LossRatioCallback()]
)

\# Validações


1.   Gráfica
2.   Matriz de confusão
3.   Sumário do modelo



In [ ]:
import matplotlib.pyplot as plt

print("#########################################################################")
print("Gráfica")
# Plotar a perda (loss)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Perda de Treinamento')
plt.plot(history.history['val_loss'], label='Perda de Validação')
plt.title('Histórico de Perda do Modelo')
plt.xlabel('Época')
plt.ylabel('Perda')
plt.legend()
plt.grid(True)

# Plotar a acurácia
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Acurácia de Treinamento')
plt.plot(history.history['val_accuracy'], label='Acurácia de Validação')
plt.title('Histórico de Acurácia do Modelo')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print("#########################################################################")
print("Matriz de confusão")
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Get predictions for the test set
test_predictions_raw = model.predict(X_test)
test_predictions_classes = np.argmax(test_predictions_raw, axis=1)

# Calculate the confusion matrix
cm = confusion_matrix(y_test, test_predictions_classes, labels=range(len(CLASSES)))

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

print("#########################################################################")
print("Sumário do modelo")
model.summary()



In [ ]:
def select_file():# Filter list to include only files
  # Uses test submission files
  dir_test = os.path.join(path, "Test_submission", "Test_submission")
  metdata_test_file=os.path.join(path, "Metadata_Test.csv")


  # Select random file
  files = [f for f in os.listdir(dir_test) if os.path.isfile(os.path.join(dir_test, f))]
  if files:
      random_file = random.choice(files)
      #print(f"Selected: {random_file}\n")
      # Filter rows where a specific column matches a regex pattern
      df=pd.read_csv(metdata_test_file)
      matched_rows = df[df['FileName'].str.contains(random_file, na=False, regex=True)]
      # Print the resulting rows
      #print(matched_rows,"\n")
      random_file = os.path.join(dir_test, random_file)
  else:
      print("No files found in directory.")


  if not os.path.isfile(random_file):
      print(f"Arquivo não encontrado: {random_file}")
      os.sys.exit(1)
  return random_file, matched_rows



In [ ]:
import pandas as pd

all_results = [] # Lista para armazenar os resultados de cada iteração

for i in range(10): # Loop 10 vezes
  print(f"\n--- Iteração {i+1}/10 ---")
  random_file, matched_rows = select_file() # Chamar a função para selecionar um novo arquivo e metadados
  print(f"Processando o arquivo de áudio: {os.path.basename(random_file)}")

  try:
      # `audio_to_spectrogram` já converte para o tamanho e 3 canais
      spectrogram_image = audio_to_spectrogram(random_file)
      # Adiciona a dimensão do batch para o modelo
      input_spectrogram = np.expand_dims(spectrogram_image, axis=0)

      # 2. Make a prediction using the trained model
      predictions = model.predict(input_spectrogram)
      predicted_class_index = np.argmax(predictions)
      predicted_class_label = CLASSES[predicted_class_index]

      # 3. Get the true class label from matched_rows
      if not matched_rows.empty and 'Class' in matched_rows.columns:
          true_class_label = matched_rows['Class'].iloc[0]
      else:
          true_class_label = "Unknown (metadata not found)"
          print("Warning: Could not find true class label for the selected file.")

      # 4. Compare prediction with true class
      is_correct = (predicted_class_label == true_class_label)
      result = "Correct" if is_correct else "Incorrect"

      # 5. Store results in a DataFrame (for this iteration)
      iteration_results_df = pd.DataFrame({
          'File Name': [os.path.basename(random_file)],
          'True Class': [true_class_label],
          'Predicted Class': [predicted_class_label],
          'Prediction Result': [result]
      })
      all_results.append(iteration_results_df)

      # Print immediate feedback for each iteration
      if not is_correct:
          print(f"Oh não! A predição para '{os.path.basename(random_file)}' está INCORRETA.")
          print(f"O modelo previu '{predicted_class_label}' mas a classe verdadeira é '{true_class_label}'.")
      else:
          print(f"Excelente! A predição para '{os.path.basename(random_file)}' está CORRETA.")

  except Exception as e:
      print(f"Erro ao processar e predizer o arquivo {os.path.basename(random_file)}: {e}")

# Concatenar todos os resultados em um único DataFrame no final
if all_results:
    final_results_df = pd.concat(all_results, ignore_index=True)
    print("\n--- Resultados Finais de Todas as Predições ---")
    display(final_results_df)
else:
    print("Nenhuma predição foi realizada com sucesso.")

print("#########################################################################")
correct_predictions = final_results_df[final_results_df['Prediction Result'] == 'Correct'].shape[0]
total_predictions = final_results_df.shape[0]

if total_predictions > 0:
    accuracy = correct_predictions / total_predictions
    print(f"Acurácia do modelo no conjunto de teste: {accuracy:.2%}")
else:
    print("Não há predições para calcular a acurácia.")